In [2]:
def count_words(text):
    # Split the text into words using whitespace and count the length
    words = text.split()
    return len(words)

# Example usage
sample_text = "Sierra Entertainment, Inc. (formerly On-Line Systems and Sierra On-Line, Inc.) was an American video game developer and publisher founded in 1979 by Ken and Roberta Williams. The company is known for pioneering the graphic adventure game genre, including the first such game, Mystery House. It is known for its graphical adventure game series King's Quest, Space Quest, Police Quest, Gabriel Knight, Leisure Suit Larry, and Quest for Glory, and as the original publisher of Valve's Half-Life series.\nAfter seventeen years as an independent company, Sierra was acquired by CUC International in February 1996 to become part of CUC Software. However, CUC International was caught in an accounting scandal in 1998, and many of the original founders of Sierra including the Williamses left the company. Sierra remained as part of CUC Software as it was sold and renamed several times over the next few years. Sierra was formally disestablished as a company and reformed as a division of this group in August 2004. The former CUC Software group was acquired by Vivendi and branded as Vivendi Games in 2006. The Sierra division continued to operate through Vivendi Games's merger with Activision to form Activision Blizzard on July 10, 2008, but was shut down later that year. The Sierra brand was revived by Activision in 2014 to re-release former Sierra games and some independently developed games.\nCurrently, the Sierra brand is under Microsoft's ownership through its gaming division, following the acquisition of Activision Blizzard.\n\n"
print("Number of words:", count_words(sample_text))

Number of words: 242


In [2]:
pip install bs4

  Using cached bs4-0.0.2-py2.py3-none-any.whl.metadata (411 bytes)
  Using cached beautifulsoup4-4.12.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached soupsieve-2.6-py3-none-any.whl.metadata (4.6 kB)
Using cached bs4-0.0.2-py2.py3-none-any.whl (1.2 kB)
Using cached beautifulsoup4-4.12.3-py3-none-any.whl (147 kB)
Using cached soupsieve-2.6-py3-none-any.whl (36 kB)
Note: you may need to restart the kernel to use updated packages.


In [5]:
import requests
from bs4 import BeautifulSoup
from transformers import pipeline

def fetch_url_content(url):
    """
    Fetch and parse content from the URL.
    Parameters:
        url (str): The URL to fetch content from.
    Returns:
        str: Parsed content from the webpage.
    """
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise an exception for HTTP errors
        
        # Parse the content with BeautifulSoup
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # Extract text from <p> tags
        paragraphs = soup.find_all('p')
        content = " ".join([para.get_text() for para in paragraphs])
        
        return content.strip()
    except requests.exceptions.RequestException as e:
        print(f"Error fetching the URL: {e}")
        return ""

def chunk_text(text, max_tokens=500):
    """
    Splits text into chunks of a specified maximum token size.
    Parameters:
        text (str): The input text.
        max_tokens (int): The maximum number of tokens per chunk.
    Returns:
        list: A list of text chunks.
    """
    words = text.split()
    for i in range(0, len(words), max_tokens):
        yield " ".join(words[i:i + max_tokens])

def generate_dynamic_summary(content, query, device=0):
    """
    Generate a dynamic summary based on the input content and query.
    Parameters:
        content (str): The content from the URL.
        query (str): The user-provided query.
        device (int): The device to use (0 for GPU, -1 for CPU).
    Returns:
        str: The dynamically generated summary.
    """
    # Load a summarization pipeline with GPU support
    summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device=device)
    
    # Combine the query with the content
    input_text = f"Query: {query}\n\nContent: {content}"
    
    # Chunk the input text to handle long content
    chunks = list(chunk_text(input_text, max_tokens=500))
    
    summaries = []
    for chunk in chunks:
        try:
            summary = summarizer(chunk, max_length=150, min_length=40, do_sample=False)
            summaries.append(summary[0]['summary_text'])
        except Exception as e:
            print(f"Error generating summary for chunk: {e}")
            continue

    # Combine all summarized chunks into a final summary
    return " ".join(summaries)

# Example Usage
json_document = {
    "Title": "Civics",
    "URL": "https://en.wikipedia.org/wiki/Civics",
    "Content": "...",  # Skipping for brevity
    "Revision ID": 1255558000,
    "Topic": "Education"
}

query = "What is civic education and who should study it?"

# Fetch dynamic content from the URL
url_content = fetch_url_content(json_document["URL"])

# Generate a dynamic summary
if url_content:
    summary = generate_dynamic_summary(url_content, query, device=0)
    print("Dynamic Summary:")
    print(summary)
else:
    print("Failed to fetch content from the URL.")


Your max_length is set to 150, but your input_length is only 7. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=3)


Dynamic Summary:
Civics is the study of the civil and political rights and obligations of citizens in a society. In U.S. politics, in the context of urban planning, the term civics comprehends the city politics that affect the political decisions of the citizenry of a city. In Ancient Rome, civics also refers to the Civic Crown, which was a garland of oak leaves awarded to Romans who saved the lives of fellow citizens. The Spartan ideal of civic education was a process whereby the citizen becomes totally united with the interest of the polity, in a spirit of perfect patriotism. Plutarch recounts how Lycurgus 'ordered the maidens to exercises themselves with wrestling, running, throwing the quoit, and chasing the dart' with a view to creating healthy children for the state. Plutarch spoke of the influence of Homer's 'lessons of state' on Lycurgus, framer of the Spartan constitution. In the Euripides tragedy The Suppliants, King Adrastus of Argos describes how Hippomedon received his civ

In [ ]:
import json

# Load documents
with open("all_topics_wikipedia_data.json", "r") as f:
    documents = json.load(f)

# Create a mapping from doc_id to URL
doc_id_to_url = {}
for topic_docs in documents.values():
    for doc in topic_docs:
        doc_id = int(doc.get("Revision ID", -1))  # Use Revision ID as doc_id
        if doc_id != -1:
            doc_id_to_url[doc_id] = doc.get("URL", "")

# Save the mapping to a file
with open("doc_id_to_url.json", "w") as f:
    json.dump(doc_id_to_url, f)
print("doc_id_to_url mapping saved!")


: 

Civics is the study of the civil and political rights and obligations of citizens in a society. In U.S. politics, in the context of urban planning, the term civics comprehends the city politics that affect the political decisions of the citizenry of a city. In Ancient Rome, civics also refers to the Civic Crown, which was a garland of oak leaves awarded to Romans who saved the lives of fellow citizens. The Spartan ideal of civic education was a process whereby the citizen becomes totally united with the interest of the polity, in a spirit of perfect patriotism. Plutarch recounts how Lycurgus 'ordered the maidens to exercises themselves with wrestling, running, throwing the quoit, and chasing the dart' with a view to creating healthy children for the state. Pericles' Funeral Oration provides insight into Athens' sharply contrasting form of education from Sparta. Homer's 'lessons of state' influenced Lycurgus, framer of the Spartan constitution. In the Euripides tragedy The Suppliants, King Adrastus of Argos describes how Hippomedon received his civic education for endurance, martial skill, and service to the state. Marcus Aurelius was educated as a citizen to value free speech. Aurelius of Rome was taught by his father how to live as a public figure restrained by modesty. Thomas Hobbes was deeply uncomfortable with Aristotelian civic education, which he said advised popular governance instead of monarchical rule. Francis Bacon was aware of the relevance of civic education to what he termed 'civil merit' Of consequence. of consequence of consequence. Of consequence of consequences of consequence, of consequence to consequence

Civics is the study of the civil and political rights and obligations of citizens in a society. In U.S. politics, in the context of urban planning, the term civics comprehends the city politics that affect the political decisions of the citizenry of a city. In Ancient Rome, civics also refers to the Civic Crown, which was a garland of oak leaves awarded to Romans who saved the lives of fellow citizens. The Spartan ideal of civic education was a process whereby the citizen becomes totally united with the interest of the polity, in a spirit of perfect patriotism. Plutarch recounts how Lycurgus 'ordered the maidens to exercises themselves with wrestling, running, throwing the quoit, and chasing the dart' with a view to creating healthy children for the state. Plutarch spoke of the influence of Homer's 'lessons of state' on Lycurgus, framer of the Spartan constitution. In the Euripides tragedy The Suppliants, King Adrastus of Argos describes how Hippomedon received his civic education for endurance, martial skill, and service to the state. In his Meditations, Marcus Aurelius tells of how he was educated as a citizen to value free speech. Aurelius was taught by his father how to live as a public figure restrained by modesty. Thomas Hobbes was deeply uncomfortable with Aristotelian civic education, which he said advised popular governance instead of monarchical rule. Francis Bacon argued that civic education should be preceded by religious and moral education. A. social life of consequence. A look at some of the things that have happened in the last few years. A. history of the U.S. state of the union. A history of American society in the 20th century. A chronology of American history.
